In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

In [2]:
organization = pd.read_csv("datasets/org-chart-overhaul/OfficeSpace.csv")

In [3]:
organization.head(10)

,Employee Name,Manager Name
0,Bill Lumbergh,NaN
1,Bob Slydell,Bill Lumbergh
2,Bob Porter,Bill Lumbergh
3,Linda M. Grayson,Bill Lumbergh
4,Dom Portwood,Linda M. Grayson
5,Wendy L. Hargrove,Linda M. Grayson
6,Tom Smykowski,Linda M. Grayson
7,Nathan R. Ross,Linda M. Grayson
8,Peter Gibbons,Dom Portwood
9,Cheryl T. Ackerman,Peter Gibbons


In [4]:
graph = defaultdict(list)

for emp, mgr in organization.itertuples(index=False):
    graph[mgr].append(emp)

In [5]:
graph

defaultdict(list,
            {nan: ['Bill Lumbergh'],
             'Bill Lumbergh': ['Bob Slydell',
              'Bob Porter',
              'Linda M. Grayson'],
             'Linda M. Grayson': ['Dom Portwood',
              'Wendy L. Hargrove',
              'Tom Smykowski',
              'Nathan R. Ross'],
             'Dom Portwood': ['Peter Gibbons',
              'Samir Nagheenanajar',
              'Nathan C. Carter'],
             'Peter Gibbons': ['Cheryl T. Ackerman'],
             'Samir Nagheenanajar': ['Michael Bolton'],
             'Nathan C. Carter': ['Lydia Bennett',
              'Greg S. Torres',
              'Alice L. Munroe'],
             'Wendy L. Hargrove': ['Peggy Carlson',
              'Anne Martinez',
              'Fred Wilkinson'],
             'Tom Smykowski': ['Derek P. Phillips', 'Milton Waddams'],
             'Derek P. Phillips': ['Sarah J. Greene'],
             'Nathan R. Ross': ['Alan B. Peterson'],
             'Alan B. Peterson': ['Maria D. Sa

In [7]:
report_hier = {}
direct_reports = defaultdict(int)
total_reports = defaultdict(int)

def dfs(manager):
    for employee in graph[manager]:
        direct_reports[employee] = 0
        report_hier[employee] = "{} > {}".format(report_hier[manager], employee)
        dfs(employee)
        direct_reports[manager] += 1
        total_reports[manager] += total_reports[employee] + 1

root_emp = graph[np.nan][0]
report_hier[root_emp] = root_emp
dfs(root_emp)

In [9]:
report_hier = pd.Series(report_hier, name='Report Hierarchy')
direct_reports = pd.Series(direct_reports, name='Direct Reports')
total_reports = pd.Series(total_reports, name='Total Reports')

result = (
    organization
    .merge(report_hier, how='left', left_on='Employee Name', right_index=True)
    .merge(direct_reports, how='left', left_on='Employee Name', right_index=True)
    .merge(total_reports, how='left', left_on='Employee Name', right_index=True)
)
result

,Employee Name,Manager Name,Report Hierarchy,Direct Reports,Total Reports
0,Bill Lumbergh,NaN,Bill Lumbergh,3,24
1,Bob Slydell,Bill Lumbergh,Bill Lumbergh > Bob Slydell,0,0
2,Bob Porter,Bill Lumbergh,Bill Lumbergh > Bob Porter,0,0
3,Linda M. Grayson,Bill Lumbergh,Bill Lumbergh > Linda M. Grayson,4,21
4,Dom Portwood,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Dom Portwood,3,8
5,Wendy L. Hargrove,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Wendy L. Ha...,3,3
6,Tom Smykowski,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Tom Smykowski,2,3
7,Nathan R. Ross,Linda M. Grayson,Bill Lumbergh > Linda M. Grayson > Nathan R. Ross,1,3
8,Peter Gibbons,Dom Portwood,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,1,1
9,Cheryl T. Ackerman,Peter Gibbons,Bill Lumbergh > Linda M. Grayson > Dom Portwoo...,0,0


## What is the sum of the "Total Reports" column?

In [10]:
result['Total Reports'].sum()

70